# GreenTravel Intelligence Challenge: Peak Performance ML Solution
This notebook contains the complete pipeline for predicting high-carbon business trips, including advanced feature engineering, hyperparameter optimization, and ensemble modeling.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import optuna
import json
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading

In [ ]:
train_trip = pd.read_csv('public_trip_data.csv')
train_log = pd.read_csv('public_trip_event_log.csv')
train_attr = pd.read_csv('public_trip_event_attributes.csv')

test_trip = pd.read_csv('private_trip_data.csv')
test_log = pd.read_csv('private_trip_event_log.csv')
test_attr = pd.read_csv('private_trip_event_attributes.csv')

## 2. Advanced Feature Engineering

In [ ]:
def engineer_features(df_trip, df_log, df_attr, is_train=True):
    # Event Log Aggregations
    df_log['EventTimestamp'] = pd.to_datetime(df_log['EventTimestamp'])
    log_agg = df_log.groupby('TripID').agg(
        step_count=('StepOrder', 'max'),
        unique_events=('EventName', 'nunique'),
        first_event_time=('EventTimestamp', 'min'),
        last_event_time=('EventTimestamp', 'max'),
    ).reset_index()
    log_agg['total_event_duration_hrs'] = (log_agg['last_event_time'] - log_agg['first_event_time']).dt.total_seconds() / 3600
    
    # Event counts
    event_counts = df_log.pivot_table(index='TripID', columns='EventName', values='StepOrder', aggfunc='count', fill_value=0).reset_index()
    event_counts.columns = ['TripID'] + [f'count_{c.replace(" ", "_")}' for c in event_counts.columns[1:]]
    
    # Attributes Features
    df_attr['TransportationPriceDifference'] = pd.to_numeric(df_attr['TransportationPriceDifference'], errors='coerce').fillna(0)
    df_attr['ExtensionLength'] = pd.to_numeric(df_attr['ExtensionLength'], errors='coerce').fillna(0)
    attr_feats = df_attr[['TripID', 'DaysPreapproved', 'TransportationPriceDifference', 'ExtensionLength', 'ProcessCode']].copy()
    attr_feats['has_denial'] = df_attr['ExpenseDenialReason'].notnull().astype(int)
    attr_feats['has_new_transport'] = df_attr['NewTransportSelection'].notnull().astype(int)
    attr_feats['has_new_hotel'] = df_attr['NewHotelSelection'].notnull().astype(int)
    
    # Trip Data Features
    df = df_trip.copy()
    df['route'] = df['DepartureLocationCountry'] + "_" + df['ArrivalLocationCountry']
    df['is_international'] = (df['DepartureLocationCountry'] != df['ArrivalLocationCountry']).astype(int)
    df['cost_per_night'] = df['NetCosts'] / (df['HotelNights'] + 1)
    
    # Merge
    df = df.merge(log_agg.drop(columns=['first_event_time', 'last_event_time']), on='TripID', how='left')
    df = df.merge(event_counts, on='TripID', how='left')
    df = df.merge(attr_feats, on='TripID', how='left')
    
    # Drop target leakage
    if is_train:
        leakage_cols = ['Departure_CO2e', 'Return_CO2e', 'Hotel_CO2e', 'Spend_CO2e', 'TotalCO2e', 'EmployeeNumber']
        df = df.drop(columns=[col for col in leakage_cols if col in df.columns])
    
    # Drop high-cardinality
    df = df.drop(columns=['DepartureLocationCity', 'ArrivalLocationCity', 'EntitiyCode'], errors='ignore')
    return df

train_df = engineer_features(train_trip, train_log, train_attr, is_train=True)
test_df = engineer_features(test_trip, test_log, test_attr, is_train=False)

# Encoding
cat_cols = ['DepartureLocationCountry', 'ArrivalLocationCountry', 'ShippingTypeDescription', 
            'Purpose', 'OutOfPolicy', 'BusinessUnit', 'ProcessCode', 'route', 'ShippingType']

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train_df[col].astype(str), test_df[col].astype(str)], axis=0)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))

## 3. Hyperparameter Tuning (LightGBM)

In [ ]:
X = train_df.drop(columns=['TripID', 'HighCarbon'])
y = train_df['HighCarbon']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

def objective(trial):
    param = {
        'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'boosting_type': 'gbdt',
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    gbm = lgb.LGBMClassifier(**param)
    gbm.fit(X_train, y_train)
    return roc_auc_score(y_val, gbm.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # Reduced for notebook execution speed
best_params = study.best_params

## 4. Cross-Validation & Ensemble Modeling

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_test = test_df.drop(columns=['TripID'])
lgbm_preds = np.zeros(len(X_test))
xgb_preds = np.zeros(len(X_test))
cat_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
    y_t, y_v = y.iloc[train_idx], y.iloc[val_idx]
    
    # LightGBM
    m_lgb = lgb.LGBMClassifier(**best_params, random_state=42, verbose=-1)
    m_lgb.fit(X_t, y_t)
    lgbm_preds += m_lgb.predict_proba(X_test)[:, 1] / 5
    
    # XGBoost
    m_xgb = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
    m_xgb.fit(X_t, y_t)
    xgb_preds += m_xgb.predict_proba(X_test)[:, 1] / 5
    
    # CatBoost
    m_cat = CatBoostClassifier(random_state=42, verbose=0)
    m_cat.fit(X_t, y_t)
    cat_preds += m_cat.predict_proba(X_test)[:, 1] / 5

final_probs = (lgbm_preds + xgb_preds + cat_preds) / 3

## 5. Final Submission

In [ ]:
submission = pd.DataFrame({'TripID': test_df['TripID'], 'HighCarbon': final_probs})
submission.to_csv('submission.csv', index=False)
print("Submission file generated successfully.")